# 04 – Data Manipulation with NumPy & Pandas (S4)

**Course section:** NumPy & Pandas.

**Goal:** practise the core NumPy / Pandas operations on the New York listings and prepare an enriched table for EDA (S5):
arrays and vectorised maths → Series/DataFrame and dtypes → selection (`loc`, `iloc`, `query`) → conditional operations → `groupby` with `agg`, `filter`, `transform`, `apply` → pivot tables and crosstabs → joins with the review data → reading an external file.

**Rules kept from S2:** price statistics use the regression subset (`price > 0`); review-based columns are created for **EDA only** and are not model features (D7). No rows are dropped from the data.

In [1]:
# --- Imports and settings ---
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))
from configs.config import CITY, RANDOM_STATE, TARGET
from src.data_utils import load_city_listings, parse_amenities, to_bool

RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"; PROCESSED.mkdir(parents=True, exist_ok=True)
TABLE_DIR = PROJECT_ROOT / "reports" / "tables"; TABLE_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)

df = to_bool(load_city_listings(CITY))          # t/f columns -> boolean
print(df.shape)

(37012, 33)


## 1. NumPy: arrays and vectorised operations

In [2]:
price = df[TARGET].to_numpy()                    # pandas -> NumPy array
print(type(price), price.dtype, price.shape, "| ndim:", price.ndim)

pos = price[price > 0]                           # boolean-mask indexing
print("Positive prices:", pos.size)
print("Percentiles 10/50/90:", np.percentile(pos, [10, 50, 90]))
print("Mean of log1p(price):", np.log1p(pos).mean().round(3))   # vectorised: no Python loop

# np.where: a quick price band, and a reproducible random sample of 5 prices
band = np.where(pos >= 200, "high", np.where(pos >= 80, "mid", "low"))
print(dict(zip(*np.unique(band, return_counts=True))))
rng = np.random.default_rng(RANDOM_STATE)
print("Random sample:", rng.choice(pos, size=5, replace=False))

<class 'numpy.ndarray'> int64 (37012,) | ndim: 1
Positive prices: 36984
Percentiles 10/50/90: [ 42.  99. 250.]
Mean of log1p(price): 4.631
{'high': 5921, 'low': 14062, 'mid': 17001}
Random sample: [210  40  49 154 190]


In [3]:
# A 2-D array: guests vs. bedrooms for listings where both are known
arr = df[["accommodates", "bedrooms"]].dropna().to_numpy()
print("shape:", arr.shape, "| column means:", arr.mean(axis=0).round(2))
print("guests per bedroom (median):", np.median(arr[:, 0] / arr[:, 1]).round(2))

shape: (33404, 2) | column means: [2.83 1.32]
guests per bedroom (median): 2.0


## 2. Series, DataFrame and data types
Right dtypes save memory and unlock date and category operations.

In [4]:
before = df.memory_usage(deep=True).sum() / 1e6

df["host_since"] = pd.to_datetime(df["host_since"])
cat_cols = ["district", "room_type", "property_type", "neighbourhood", "host_response_time"]
df[cat_cols] = df[cat_cols].astype("category")

after = df.memory_usage(deep=True).sum() / 1e6
print(f"Memory: {before:.1f} MB -> {after:.1f} MB")
df.dtypes.value_counts()

Memory: 43.7 MB -> 29.5 MB


float64           13
int64              6
str                4
boolean            4
datetime64[us]     1
category           1
category           1
category           1
category           1
category           1
Name: count, dtype: int64

In [5]:
# Series basics: index, values, a summary
s = df.set_index("listing_id")[TARGET]
print(type(s).__name__, "| index name:", s.index.name, "| length:", len(s))
print("First three:", s.head(3).to_dict())
print("Most expensive listing_id:", s.idxmax(), "->", s.max(), "USD")

Series | index name: listing_id | length: 37012
First three: {3457770: 110, 4173751: 99, 4259427: 220}
Most expensive listing_id: 13925864 -> 10000 USD


## 3. Selection: `loc`, `iloc`, `query`

In [6]:
print(df.iloc[:3, :4])                                         # by position
print()
print(df.loc[df["district"] == "Staten Island", ["neighbourhood", "room_type", TARGET]].head(3))   # by label + condition
print()
cheap_entire = df.query("room_type == 'Entire place' and price > 0 and price < 60")
print("Entire places under 60 USD:", len(cheap_entire))

   listing_id                                 name   host_id host_since
0     3457770     Cute 1 BR in the Lower East Side   9604972 2013-10-23
1     4173751      Lovely Apartment in Harlem West   9197440 2013-10-02
2     4259427  Appartement en duplex de 3 chambres  21833312 2014-09-27

    neighbourhood     room_type  price
546    St. George  Entire place     89
853  Prince's Bay  Entire place    145
861      New Dorp  Entire place     40

Entire places under 60 USD: 461


## 4. Conditional operations

In [7]:
pos_df = df[df[TARGET] > 0].copy()                 # regression subset (D4), not a deletion from df

# isin / between
print("Manhattan or Brooklyn:", pos_df["district"].isin(["Manhattan", "Brooklyn"]).mean().round(3))
print("Price 50-150 USD     :", pos_df[TARGET].between(50, 150).mean().round(3))

# np.select: stay type from minimum_nights
conditions = [pos_df["minimum_nights"] >= 30, pos_df["minimum_nights"] >= 7]
pos_df["stay_type"] = np.select(conditions, ["monthly", "weekly"], default="short")

# Price per guest (guarding against accommodates == 0)
pos_df["price_per_guest"] = pos_df[TARGET] / pos_df["accommodates"].where(pos_df["accommodates"] > 0)

pos_df.groupby("stay_type")[[TARGET, "price_per_guest"]].median().round(1)

Manhattan or Brooklyn: 0.838
Price 50-150 USD     : 0.601


,price,price_per_guest
stay_type,,
monthly,99.0,43.3
short,95.0,37.5
weekly,94.0,41.7


## 5. `groupby` + `agg`: borough × room type

In [8]:
borough_room = (pos_df.groupby(["district", "room_type"], observed=True)
                      .agg(n_listings=("listing_id", "size"),
                           median_price=(TARGET, "median"),
                           mean_guests=("accommodates", "mean"),
                           superhost_share=("host_is_superhost", "mean"))
                      .round(2))
borough_room.to_csv(TABLE_DIR / "04_borough_roomtype.csv")
borough_room

n_listings  median_price  mean_guests  superhost_share
district      room_type                                                           
Bronx         Entire place         368         102.5         4.29             0.28
              Private room         592          50.5         1.98             0.28
              Shared room           31          28.0         1.48              0.1
Brooklyn      Entire place        7206         130.0         3.82             0.23
              Hotel room            15         119.0         2.27             0.27
              Private room        7021          57.0         1.85             0.17
              Shared room          230          35.0         1.87             0.19
Manhattan     Entire place       10014         150.0         3.35             0.16
              Hotel room           247         179.0         2.82             0.15
              Private room        5975          77.0         1.89             0.16
              Shared room          292          62.5         1.79             0.14
Queens        Entire place        1662         111.0         4.19             0.29
              Hotel room             9         109.0         2.11              0.0
              Private room        2903          50.0         1.89             0.18
              Shared room          130          43.0         1.75             0.15
Staten Island Entire place         147         115.0         4.59             0.42
              Private room         139          59.0         1.91              0.4
              Shared room            3          79.0         2.00             0.33

## 6. `filter`: keep neighbourhoods with enough listings
Medians from tiny groups are unreliable, so neighbourhood comparisons use only groups with ≥ 100 listings.

In [9]:
big = pos_df.groupby("neighbourhood", observed=True).filter(lambda g: len(g) >= 100)
print("Neighbourhoods total:", pos_df["neighbourhood"].nunique(),
      "| with >= 100 listings:", big["neighbourhood"].nunique(),
      "| share of listings covered:", round(len(big) / len(pos_df), 3))

nbh = (big.groupby("neighbourhood", observed=True)[TARGET]
          .agg(n="size", median="median")
          .sort_values("median", ascending=False))
pd.concat([nbh.head(5), nbh.tail(5)])

Neighbourhoods total: 220 | with >= 100 listings: 59 | share of listings covered: 0.89


,n,median
neighbourhood,,
Tribeca,167,209.0
Theater District,301,172.0
SoHo,280,170.0
Greenwich Village,259,169.0
West Village,598,166.5
Bushwick,1678,60.0
Ridgewood,291,59.0
Jackson Heights,213,51.0
Elmhurst,241,50.0


## 7. `transform`: price relative to the neighbourhood
`transform` returns one value per row, so each listing can be compared with its own neighbourhood.
*Note:* group statistics computed on the **whole** data are fine for EDA, but in modeling they must be computed on training folds only (leakage).

In [10]:
pos_df["nbh_median_price"] = pos_df.groupby("neighbourhood", observed=True)[TARGET].transform("median")
pos_df["price_vs_nbh"] = (pos_df[TARGET] / pos_df["nbh_median_price"]).round(2)

print(pos_df["price_vs_nbh"].describe(percentiles=[.1, .5, .9]).round(2).to_string())
print("\nListings priced at least 3x their neighbourhood median:", int((pos_df["price_vs_nbh"] >= 3).sum()))

count    36984.00
mean         1.40
std          2.61
min          0.07
10%          0.50
50%          1.00
90%          2.31
max        142.86

Listings priced at least 3x their neighbourhood median: 2081


## 8. `apply`: custom functions on rows and groups

In [11]:
pos_df["n_amenities"] = pos_df["amenities"].apply(lambda s: len(parse_amenities(s)))

# Most common property type in each borough (apply on a grouped Series)
top_type = pos_df.groupby("district", observed=True)["property_type"].apply(lambda s: s.value_counts().index[0])

# Row-wise apply with a named function
def size_class(row) -> str:
    beds = row["bedrooms"]
    if pd.isna(beds):
        return "unknown"
    return "studio/1BR" if beds <= 1 else ("2BR" if beds == 2 else "3BR+")

pos_df["size_class"] = pos_df.apply(size_class, axis=1)

print(top_type.to_string(), "\n")
pos_df.groupby("size_class")[[TARGET, "n_amenities"]].median()

district
Bronx            Private room in apartment
Brooklyn                  Entire apartment
Manhattan                 Entire apartment
Queens           Private room in apartment
Staten Island        Private room in house 



,price,n_amenities
size_class,,
2BR,152.0,22.0
3BR+,230.5,24.0
studio/1BR,80.0,18.0
unknown,105.0,18.0


## 9. Pivot tables and crosstabs

In [12]:
pivot = pd.pivot_table(pos_df, values=TARGET, index="district", columns="room_type",
                       aggfunc="median", observed=True)
pivot.to_csv(TABLE_DIR / "04_pivot_median_price.csv")
pivot

room_type,Entire place,Hotel room,Private room,Shared room
district,,,,
Bronx,102.5,NaN,50.5,28.0
Brooklyn,130.0,119.0,57.0,35.0
Manhattan,150.0,179.0,77.0,62.5
Queens,111.0,109.0,50.0,43.0
Staten Island,115.0,NaN,59.0,79.0


In [13]:
# Share of Superhosts by borough and room type (row-normalised crosstab)
pd.crosstab(pos_df["district"], pos_df["room_type"],
            values=pos_df["host_is_superhost"].astype("float"), aggfunc="mean").round(3)

room_type,Entire place,Hotel room,Private room,Shared room
district,,,,
Bronx,0.277,NaN,0.277,0.097
Brooklyn,0.231,0.267,0.175,0.187
Manhattan,0.158,0.146,0.159,0.140
Queens,0.288,0.000,0.185,0.154
Staten Island,0.422,NaN,0.396,0.333


## 10. Joins: listings + review activity (EDA only)
We aggregate the 5.4 million review rows to **one row per listing** and join them to the listings with a left join.
These columns describe review activity for EDA and S6; they are **not** model features for price (D7).

In [14]:
reviews = pd.read_csv(RAW / "Reviews.csv", usecols=["listing_id", "date"],
                      encoding="latin-1", parse_dates=["date"])
snapshot = reviews.loc[reviews["listing_id"].isin(df["listing_id"]), "date"].max()
print("All reviews:", len(reviews), "| NY snapshot date (last NY review):", snapshot.date())

last_year = reviews["date"] > snapshot - pd.Timedelta(days=365)
rev_agg = (reviews.assign(last_12m=last_year)
                  .groupby("listing_id")
                  .agg(n_reviews=("date", "size"),
                       first_review=("date", "min"),
                       last_review=("date", "max"),
                       reviews_last_12m=("last_12m", "sum")))

enriched = df.merge(rev_agg, on="listing_id", how="left", validate="one_to_one", indicator=True)
print(enriched["_merge"].value_counts().to_string())

enriched[["n_reviews", "reviews_last_12m"]] = enriched[["n_reviews", "reviews_last_12m"]].fillna(0).astype(int)
enriched["host_tenure_years"] = ((snapshot - enriched["host_since"]).dt.days / 365.25).round(1)
enriched = enriched.drop(columns="_merge")
print(enriched[["n_reviews", "reviews_last_12m", "host_tenure_years"]].describe().round(1).to_string())

All reviews: 5373143 | NY snapshot date (last NY review): 2021-02-05
_merge
both          27489
left_only      9523
right_only        0
       n_reviews  reviews_last_12m  host_tenure_years
count    37012.0           37012.0            36994.0
mean        22.9               2.7                5.3
std         47.9               9.6                2.6
min          0.0               0.0                0.0
25%          0.0               0.0                3.3
50%          4.0               0.0                5.5
75%         20.0               2.0                7.3
max        753.0             514.0               12.5


In [15]:
# Consistency check: listings without any review should be exactly the ones without review scores
no_rev = enriched["n_reviews"] == 0
pd.crosstab(no_rev, enriched["review_scores_rating"].isna(),
            rownames=["no reviews"], colnames=["rating missing"])

rating missing,False,True
no reviews,,
False,26777,712
True,0,9523


## 11. Reading an external file and joining it: the dataset's own data dictionary

In [16]:
official = pd.read_csv(RAW / "Listings_data_dictionary.csv", encoding="latin-1")
ours = pd.read_csv(TABLE_DIR / "02_variable_dictionary.csv")

joined = ours.merge(official, left_on="variable", right_on="Field", how="outer", indicator=True)
print(joined["_merge"].value_counts().to_string())
joined.loc[joined["_merge"] != "both", ["variable", "Field", "_merge"]]

_merge
both          33
left_only      0
right_only     0


,variable,Field,_merge


## 12. Save the enriched table for EDA (S5)

In [17]:
out_path = PROCESSED / "ny_listings_enriched.pkl"
enriched.to_pickle(out_path)       # pickle keeps the dtypes (dates, categories, booleans)
print(out_path.name, enriched.shape, f"{out_path.stat().st_size / 1e6:.1f} MB")

ny_listings_enriched.pkl (37012, 38) 20.5 MB


## 13. Dataset story and evaluation of S4

**Dataset story** *(preliminary observations; tested formally in S6)*
1. **Location sets the price level.** Among the 59 neighbourhoods with at least 100 listings (89% of all listings), the median price ranges from **209 USD in Tribeca** to **34 USD in Woodside**, roughly 6x. Manhattan is the most expensive borough for every room type except shared rooms on Staten Island (only 3 listings).
2. **Room type is the second big driver.** In every borough an entire place costs about twice a private room (Manhattan 150 vs 77, Brooklyn 130 vs 57, Queens 111 vs 50). Size matters too: median 80 USD for studio/1-bedroom, 152 for 2 bedrooms, 230.5 for 3+.
3. **Monthly rentals are not cheaper per night.** Listings requiring 30+ nights have the same median nightly price (99 USD) as short stays (95 USD); price per guest is slightly higher (43 vs 38 USD).
4. **Superhosts are more common outside Manhattan.** The Superhost share is about 14-16% in Manhattan, 17-23% in Brooklyn (entire places and private rooms), 28% in the Bronx and around 40% on Staten Island → a candidate hypothesis for S6.
5. **Review activity collapsed during COVID.** At least half of the listings received no review in the 12 months before the snapshot (median 0, 75th percentile 2), although the median listing has 4 reviews in total.
6. **Some prices look extreme relative to their area.** 2,081 listings (5.6%) are priced at 3x or more of their neighbourhood median; the maximum ratio is 143 (the 10,000 USD listing). These are candidates for the outlier analysis in S7.

**Checks that passed**
- The review join matched 27,489 listings; the 9,523 without reviews are exactly the listings without review scores, and 712 listings have reviews but no rating yet.
- Our variable dictionary and the dataset's official dictionary match on all 33 fields.

**What we did (course coverage):** NumPy arrays, masks, `np.where`, `np.select`, seeded random sampling; dtypes (dates, categories: 43.7 → 29.5 MB); `loc` / `iloc` / `query`; `isin` / `between`; `groupby` with named `agg`, `filter`, `transform`, `apply`; `pivot_table` and `crosstab`; a validated one-to-one left join; reading an external file and an outer join.

**Outputs:** `reports/tables/04_borough_roomtype.csv`, `reports/tables/04_pivot_median_price.csv`, and `data/processed/ny_listings_enriched.pkl` (37,012 × 38, not committed; recreated by running this notebook).

**Decisions:** Tier 1 only. No rows or columns were removed; review-based columns are for EDA and S6 only (D7).

**Next step:** S5 – exploratory data analysis and visualisation, one dataset story per block.
